<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/Fine_tune_Llama_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 trl==0.4.7
!{sys.executable} -m pip install -q transformers==4.31.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
!pip install trl==0.4.7

In [2]:
!pip install --upgrade transformers trl peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
#peft means parameter efficient transfer learning inside this we will find lora etc
import os
import torch
#from datasets import load_dataset
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig,HfArgumentParser,TrainingArguments,pipeline,logging
from peft import LoraConfig,PeftModel
from trl import SFTTrainer

In [4]:
from datasets import load_dataset

In [5]:
#model that we need to train

model_name='NousResearch/Llama-2-7b-chat-hf'

#dataset

dataset_name='mlabonne/guanaco-llama2-1k'

#fine-tuned model name

new_model='Llama-2-7b-chat-finetune'

lora_r=64 #rank
lora_alpha=16 #means the effect of the fine tune data on the train model
lora_dropout=0.1 # means the 10 percent of the adapter pathways are dropped or detactivated randomly so that model might not memorise

use_4bit=True # activate 4 bit precision model
bnb_4bit_compute_dtype='float16' # but do the calculation in the 16-bit to preserve quality
bnb_4bit_quant_type='nf4' #np4 or fp4.0
use_nested_quant=False # 4 bit base models(double quantisation)

output_dir='./results'

num_train_epochs=1

fp16=True
bf16=False  # set it true when you are using A100 gpu

per_device_train_batch_size=4
per_device_eval_batch_size=4
gradient_accumalation_steps=1  # the model update the weighths after every single batch
max_grad_norm=3  # sometimes the gradient might spike which can distrub the weights so we are setting the threshold
learning_rate=2e-4
weight_decay=0.001  #penalty to reduce overfitting
optim="paged_adamw_32bit" #This is your optimizer choice, specifically tailored for saving GPU memory.

max_steps=-1  #This tells the training script not to cap the training by a specific number of individual steps.
warmup_ratio=0.03
group_by_length=True #Group sequences into batches with same length
save_steps=0 #The model will only save at the very end of training.
logging_steps=25  # every 25 training steps the log message about the loss is printed

max_seq_length=None #Maximum Sequence length to use
packing=False # pack multiple short examples in the same input sequence
device_map={"":0}  # Load the entire model on the GPU 0


In [6]:
dataset=load_dataset(dataset_name,split="train")

compute_dtype=getattr(torch,bnb_4bit_compute_dtype)

bnb_config=BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant
)

model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache=False
# model.config.pretraining_tp=1  # <-- Removed to avoid errors on newer models

tokenizer=AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'

peft_config=LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias='none',
    task_type='CAUSAL_LM'
)

training_arguments=TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumalation_steps, # <-- Double check variable name matches your setup
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    # group_by_length=group_by_length,  # <-- Removed to fix your current error
    lr_scheduler_type='constant',
    report_to='tensorboard'
)

trainer=SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

NameError: name 'gradient_accumulation_steps' is not defined